# Optimizing Agentic AI Collaboration v1.1.0 - GSM8K Full Workflow Template

## Project aims

 1. **a LangChain-based multi-agent Society-of-Minds prototype** supporting **≥3 interaction protocols**,
 2. **benchmark gains** (≥5% on **GSM8K** and **MMLU** vs a single-agent SLM baseline), and
 3. a **cost/benefit profile** using **latency + total token consumption**, all tracked in **W&B** for a claim-to-evidence table.  

## Current notebook version achievements

The code acts as a **template** for execution and evaluation of both single agent baselines and multi-agentic systems over the **GSM8K** dataset. The run is orchestrated and stored through W&B.
It covers the full workflow of the project experiments:
1. Automated ingestion of GSM8K dataset and creation of single/multi agent pipeline.
2. Evaluation of the agentic system over multiple GSM8K examples. Each sample is evaluated with 3 different seeds to reduce estimation variability.
3. Definition of statistically consitent estimators and confidence intervals for accuracy and execution times. The results and the statistical measures are stored on W&B.

However, it does **not yet produce evidence** for the thesis’ central claims (benchmark gains + compute trade-off on GSM8K/MMLU).

## Setup

**Hardware target:** Google Colab T4 GPU (free tier).
**Model:** `mistralai/Mistral-7B-Instruct-v0.2` with 4-bit NF4 quantization.

---
### Additional information

1. **LangChain-based multi-agent orchestration** — agents are defined as LangChain `LLMChain` instances with structured `PromptTemplate`s, composed via a custom sequential pipeline. This replaces ad-hoc string concatenation with a modular, extensible framework.
2. **Weights & Biases (W&B) experiment tracking** — every agent call, its latency, token counts, and full text are logged as W&B artifacts and metrics, enabling richer post-hoc analysis across runs.

---

### Before running

Complete these manual steps in Colab:

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets


## What the current notebook implements

The implementation is a **LangChain refactor** of a *single* 3-step pipeline:

* **Proposer → Critic → Refiner**, plus a **single-agent baseline**.
* **W&B logging** of per-step latency and output length; plus a W&B `Table` comparing outputs across tasks.
* Evaluation is on **MMLU random samples.**
**Observed latencies and overhead (from the notebook run outputs)**. **Accuracy and latency averages are estimated with both a point and a 95% confidence interval**
* Multiple random seeds are defined to **allow reproductibility**.


---

## Objective-by-objective diagnosis

### OBJ-1 (multi-agent prototype, ≥3 protocols)

* **Partially relevant**: there is a working multi-agent scaffold (roles, orchestration, logging).
* **Not compliant** with the “≥3 distinct interaction protocols” requirement: there is **only one protocol** (Critique→Refine). No majority voting, no self-consistency, no multi-round debate.

### OBJ-2 (≥5% accuracy gain on GSM8K + MMLU vs single-agent baseline)

* **Adressed**: dataset harness, scoring based on accuracy (with both point and interval-based estimations) compared to a single-agent baseline. Execution repeated with 3 distinct seeds to gain consistency and reduce estimations variability.

### OBJ-3 (accuracy-to-compute trade-off: latency + total token consumption)

* **Partially addressed**: latency is measured and logged. Average latency is estimated using both a point and confidence intervals.
* **Missing**: “total token consumption” (you log output chars, not tokens), and you’re not logging GPU memory / throughput in a benchmark setting.

### Scope alignment

* Model is **7B** and 4-bit quantized (within the “1B–7B” envelope), so *model class is in-scope*.
* The current artifact is able to compare with statistically consistent measures of the latency and the accuracy the model's execution. It logs all the executions through WandB
* However, the model used remains quite far away from the current state-of-the-art. Furthermore, only 20 distinct GSM8K samples are evaluated, *not enough for a consistent, low-variance estimation of the agents performance*.

---



## 1. Setup and Installation

We install all the libraries required for the Github repo.
For loading models and data, orchestrating agents, and saving the results, we use Hugging Face stack (same as v0.4), plus `langchain`, `langchain-community`, `langchain-core`, `datasets` and `wandb`.

In [1]:
# ── Google Colab: clone private repo using GITHUB_TOKEN secret ──────────
import sys, os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ

if IN_COLAB:
    from google.colab import userdata  # type: ignore
    _token = userdata.get("GITHUB_TOKEN")
    _repo_url = f"https://{_token}@github.com/andreufc/tfg-agentic-ai.git"
    _clone_dir = "/content/tfg-agentic-ai"
    if not os.path.isdir(_clone_dir):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", _repo_url, _clone_dir], check=True)
        print(f"\u2705 Repo cloned to {_clone_dir}")
    else:
        print(f"\u2139\ufe0f  Repo already present at {_clone_dir}")
    # Move working directory inside notebooks/ so relative paths work
    os.chdir(os.path.join(_clone_dir, "notebooks"))
    repo_root = _clone_dir
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [ ]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%pip install -r requirements.txt

No se esperaba & en este momento.
No se pudo analizar el valor especificado en una clave del Registro de ejecuci�n autom�tica.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Authentication via Colab Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab's built-in secrets manager — no tokens in code.


In [2]:
from utils.config import load_env

# Detect Google Colab environment
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ
print(f"Running on Google Colab: {IN_COLAB}")

load_env(in_colab=IN_COLAB)

Running on Google Colab: False
✅ Secrets loaded from .env file.


## 3. Initialize Weights & Biases

We create a W&B run that will persist for the entire notebook session.  All agent calls, latencies, and outputs will be logged here.


In [ ]:
from utils.logging_utils import init_wandb

project = "agentic-collab-setup" if IN_COLAB else "dev-agentic-collab-setup"
print(f"W&B Project: {project}")
run = init_wandb(
    run_name="GSM8K-Evaluation-Mistral-7B-Instruct",
    project=project,
    config={
        "model_id":       "mistralai/Mistral-7B-Instruct-v0.2",
        "quantization":   "nf4-4bit",
        "framework":      "langchain",
        "agents":         ["proposer", "critic", "refiner"],
        "max_new_tokens": {"proposer": 1024, "critic": 1024, "refiner": 1024, "baseline": 512},
        "temperature":    0.7,
        "top_p":          0.95,
        "top_k":          50,
        "eval_dataset":   "gsm8k",
        "eval_size":      20,
        "data_seed":    42,
        "generation_seeds": [42, 123, 456],
    },
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


W&B Project: dev-agentic-collab


wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B run: basic-GSM8K-evaluation  —  https://wandb.ai/tfg-optimizing-agentic-ai/dev-agentic-collab/runs/0iebd45b


## 4. Data Loading
We load the GSM8K (Grade School Math tasks) dataset, which will be used later to evaluate the agentic workflows.


In [4]:
from utils.data import load_gsm8k

# ── GSM8K DATASET ──────────────────────────────────────────────────
gsm8k_dataset = load_gsm8k(split="test")

print(f"✅ GSM8K dataset loaded with {len(gsm8k_dataset)} examples.")
print(f"Example question: {gsm8k_dataset[0]['question']}")
print(f"Example answer:  {gsm8k_dataset[0]['answer']}")

c:\Users\andreu\OneDrive\Escritorio\GCED\TFG\tfg-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 113901.03 examples/s]

✅ GSM8K dataset loaded with 1319 examples.
Example question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Example answer:  Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


## 5. Model Loading (4-bit Quantized)

Identical to v0.4: we load `Mistral-7B-Instruct-v0.2` in NF4 4-bit via `bitsandbytes`, which keeps VRAM usage under ~6 GB on a T4.


In [ ]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = wandb.config.model_id
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

## 6. LangChain LLM Wrapper

We wrap the local HF model in a `HuggingFacePipeline` so it can be used as the backend for LangChain chains.  This gives us `PromptTemplate` composition, chain sequencing, and callback hooks — all of which we exploit below.


In [ ]:
from utils.agents import build_hf_langchain_llm

try:
  assert "generation_seeds" in wandb.config and type(wandb.config.generation_seeds) == int
  generation_seed = wandb.config.generation_seeds
except:
  generation_seed = None
# Wrap the loaded model as a LangChain-compatible LLM
llm = build_hf_langchain_llm(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=wandb.config.max_new_tokens["proposer"],
    temperature=wandb.config.temperature,
    top_k=wandb.config.top_k,
    top_p=wandb.config.top_p,
    return_full_text=False,
    seed=generation_seed,
)

# Quick sanity check
print(llm.invoke("What is the capital of France?")[:200])

## 7. Defining Agents as LangChain Chains

Each agent is a `LLMChain` built from a `PromptTemplate`.  This is a clean separation of *role definition* (the template) from *execution engine* (the LLM), making it trivial to swap models, add agents, or modify prompts independently.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Parser
parser = StrOutputParser()

# ── Proposer ──────────────────────────────────────────────────
proposer_template = PromptTemplate.from_template(

        "You are the Proposer Agent. Your role is to analyze the "
        "following task and provide a clear, step-by-step solution. "
        "Be detailed in your reasoning.\n\n"
        "Task: {task}\n\n"
        "Provide your proposed solution below:"
)
proposer_chain = proposer_template | llm | parser

# ── Critic ────────────────────────────────────────────────────
critic_template = PromptTemplate.from_template(
        "You are the Critic Agent. Your role is to meticulously "
        "review the proposed solution for the given task. Identify "
        "any potential flaws, logical errors, missed assumptions, "
        "or areas for improvement. Do not provide a new solution, "
        "only critique the existing one.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution to review:\n---\n{proposal}\n---\n\n"
        "Provide your critical analysis below:"
)
critic_chain = critic_template | llm | parser

# ── Refiner ───────────────────────────────────────────────────
refiner_template = PromptTemplate.from_template(
        "You are the Refiner Agent. Your role is to produce a final, "
        "correct solution to a task using an initial proposal and a "
        "critique of that proposal. Carefully consider the critique "
        "and use it to fix the flaws in the original proposal.\n\n"
        "Original Task: {task}\n\n"
        "Proposed Solution:\n---\n{proposal}\n---\n\n"
        "Critique of the Solution:\n---\n{critique}\n---\n\n"
        "Based on the critique, provide the final, corrected solution "
        "below. Explain your steps clearly."
)
refiner_chain = refiner_template | llm | parser

# ── Single-Agent Baseline ────────────────────────────────────
baseline_template = PromptTemplate.from_template(
        "You are an AI assistant. Solve the following task, "
        "providing your final answer and reasoning.\n\n"
        "Task: {task} \n\n"
)
baseline_chain = baseline_template | llm | parser

print("✅ All agent chains defined.")

✅ All agent chains defined.


## 8. Multi-Agent Orchestrator with W&B Logging

The `run_multi_agent` function sequences the three chains and logs per-step metrics to W&B.  It also captures a W&B `Table` for side-by-side comparison.

Key metrics logged per step:
- `latency_s` — wall-clock time
- `output_chars` — proxy for token count
- Full text of each agent's output (as a W&B `Table` row)


In [ ]:
import time
from utils.logging_utils import log_metrics

def run_agent_step(chain, inputs, step_name):
    """Run a single LangChain chain, time it, and log to W&B."""
    t0 = time.perf_counter()
    # With LCEL, chain.invoke returns a string directly
    output_text = chain.invoke(inputs)
    elapsed = time.perf_counter() - t0

    # Log scalar metrics
    log_metrics({
        f"{step_name}/latency_s":    round(elapsed, 2),
        f"{step_name}/output_chars": len(output_text),
    })

    print(f"  ⏱  {step_name}: {elapsed:.1f}s  |  {len(output_text)} chars")
    return output_text, elapsed


def run_multi_agent(task, task_label="task", eval_type=None):
    """
    Execute the full Proposer → Critic → Refiner pipeline
    plus a single-agent baseline, with W&B logging.
    """
    print(f"\n{'='*60}")
    print(f"🤖 MULTI-AGENT PIPELINE  —  {task_label}")
    print(f"{'='*60}")

    # Step 1 — Proposer
    print("\n[Step 1] Proposer Agent")
    proposal, t_prop = run_agent_step(
        proposer_chain, {"task": task}, f"{task_label}/proposer"
    )
    if eval_type is None:  # When evaluating on a dataset, the output is not printed to avoid clutter in the Notebook
      print(f"\n[Proposer Output]:\n{proposal[:500]}\n{'...' if len(proposal)>500 else ''}")

    # Step 2 — Critic
    print("\n[Step 2] Critic Agent")
    critique, t_crit = run_agent_step(
        critic_chain, {"task": task, "proposal": proposal}, f"{task_label}/critic"
    )
    if eval_type is None:
      print(f"\n[Critic Output]:\n{critique[:500]}\n{'...' if len(critique)>500 else ''}")

    # Step 3 — Refiner
    print("\n[Step 3] Refiner Agent")
    refined, t_ref = run_agent_step(
        refiner_chain,
        {"task": task, "proposal": proposal, "critique": critique},
        f"{task_label}/refiner",
    )
    if eval_type is None:
      print(f"\n[Refiner Output]:\n{refined[:500]}\n{'...' if len(refined)>500 else ''}")

    # Baseline
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")
    baseline, t_base = run_agent_step(
        baseline_chain, {"task": task}, f"{task_label}/baseline"
    )
    if eval_type is None:
      print(f"\n[Baseline Output]:\n{baseline[:500]}\n{'...' if len(baseline)>500 else ''}")

    # Log total pipeline latency
    total_pipeline = t_prop + t_crit + t_ref
    log_metrics({
        f"{task_label}/total_pipeline_latency_s": round(total_pipeline, 2),
        f"{task_label}/baseline_latency_s":       round(t_base, 2),
        f"{task_label}/pipeline_overhead_ratio":   round(total_pipeline / max(t_base, 0.01), 2),
    })

    return {
        "proposal": proposal, "critique": critique,
        "refined": refined,   "baseline": baseline,
        "t_prop": t_prop, "t_crit": t_crit,
        "t_ref": t_ref, "t_base": t_base
    }

print("✅ Orchestrator ready.")

## 9. GSM8K Evaluation


In [ ]:
from utils.data import extract_gsm8k_answer, format_gsm8k_prompt
from utils.evaluation import extract_gsm8k_prediction

In [ ]:
import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

# Fix the evaluation subset once — all seed runs evaluate the same questions
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(gsm8k_dataset), sample_size)

# ── Correctness & time matrices: rows = problems, columns = seeds ──
agent_correctness_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
agent_time_matrix           = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    # Rebuild LLM and all agent chains with the current generation seed
    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens["proposer"],
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    proposer_chain = proposer_template | llm | parser
    critic_chain   = critic_template   | llm | parser
    refiner_chain  = refiner_template  | llm | parser
    baseline_chain = baseline_template | llm | parser

    results_table = wandb.Table(
        columns=["task", "proposer_output", "critic_output", "refiner_output",
                 "baseline_output", "ground_truth", "refiner_final_output", "baseline_final_output",
                 "proposer_latency", "critic_latency", "refiner_latency", "baseline_latency"]
    )

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"GSM8K seed={gen_seed} ({sample_size} samples)")):
        task              = format_gsm8k_prompt(item['question'])
        ground_truth_full = item['answer']
        true_answer       = extract_gsm8k_answer(ground_truth_full)

        output = None
        try:
            output            = run_multi_agent(task, task_label='gsm8k_sample', eval_type='gsm8k')
            agent_response    = output['refined']
            agent_answer      = extract_gsm8k_prediction(agent_response)
            baseline_response = output['baseline']
            baseline_answer   = extract_gsm8k_prediction(baseline_response)
        except Exception as e:
            print(f"Pipeline failed: {e}")
            agent_answer = baseline_answer = None
            agent_response = baseline_response = str(e)
            if output is None:
                output = {'proposal': '', 'critique': '', 'refined': agent_response,
                          'baseline': baseline_response,
                          't_prop': 0.0, 't_crit': 0.0, 't_ref': 0.0, 't_base': 0.0}

        agent_correct    = (agent_answer    == true_answer)
        baseline_correct = (baseline_answer == true_answer)

        # Store in matrices
        agent_correctness_matrix[item_idx, run_idx]    = int(agent_correct)
        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        agent_time_matrix[item_idx, run_idx]           = output['t_prop'] + output['t_crit'] + output['t_ref']
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']

        log_metrics({
            "gsm8k_sample/agents_accuracy":   int(agent_correct),
            "gsm8k_sample/baseline_accuracy": int(baseline_correct),
        })

        results_table.add_data(
            task, output['proposal'][:2000], output['critique'][:2000],
            output['refined'][:2000],  output['baseline'][:2000],
            true_answer, agent_answer, baseline_answer,
            round(output['t_prop'], 2), round(output['t_crit'], 2),
            round(output['t_ref'],  2), round(output['t_base'], 2),
        )

    # Log results table per seed
    wandb.log({f"seed_{gen_seed}/results_comparison": results_table})


## 10. Log Results Table & Finish W&B Run

Push the accumulated comparison table to W&B and close the run.  You can view the full dashboard (latency charts, text comparisons, overhead ratios) at the URL printed when the run was initialized.


In [ ]:
import tempfile
from utils.logging_utils import finish_wandb
from utils.statistics import (
    wilson_ci_clustered,
    mean_time_clustered,
    fit_time_distribution,
    print_accuracy_report,
    print_time_report,
    print_distribution_report,
)

# ── Agent accuracy statistics ─────────────────────────────────
agent_acc_stats = wilson_ci_clustered(agent_correctness_matrix)
print_accuracy_report("Agent Accuracy (GSM8K)", agent_acc_stats)

# ── Baseline accuracy statistics ──────────────────────────────
baseline_acc_stats = wilson_ci_clustered(baseline_correctness_matrix)
print_accuracy_report("Baseline Accuracy (GSM8K)", baseline_acc_stats)

# ── Agent execution time statistics ───────────────────────────
agent_time_stats = mean_time_clustered(agent_time_matrix)
print_time_report("Agent Execution Time (GSM8K)", agent_time_stats)

# ── Baseline execution time statistics ────────────────────────
baseline_time_stats = mean_time_clustered(baseline_time_matrix)
print_time_report("Baseline Execution Time (GSM8K)", baseline_time_stats)

# ── Distribution fitting ─────────────────────────────────────
agent_dist = fit_time_distribution(agent_time_matrix.ravel())
print_distribution_report("Agent Time (GSM8K)", agent_dist)

baseline_dist = fit_time_distribution(baseline_time_matrix.ravel())
print_distribution_report("Baseline Time (GSM8K)", baseline_dist)

# ── Log to W&B summary ───────────────────────────────────────
# Kept from original
wandb.summary["n_seed_runs"]                 = len(generation_seeds)
wandb.summary["total_experiments"]           = sample_size * len(generation_seeds)
wandb.summary["avg_pipeline_overhead_ratio"] = round(
    float(np.sum(agent_time_matrix)) / max(float(np.sum(baseline_time_matrix)), 0.01), 2
)

# Agent accuracy
wandb.summary["agent/accuracy"]      = round(agent_acc_stats["p_hat"], 6)
wandb.summary["agent/accuracy_ci"]   = [round(agent_acc_stats["ci_lower"], 6),
                                         round(agent_acc_stats["ci_upper"], 6)]
wandb.summary["agent/accuracy_se"]   = round(agent_acc_stats["se"], 6)
wandb.summary["agent/accuracy_deff"] = round(agent_acc_stats["deff"], 4)
wandb.summary["agent/accuracy_n_eff"] = round(agent_acc_stats["n_eff"], 2)
wandb.summary["agent/accuracy_icc"]  = round(agent_acc_stats["icc"], 4)

# Baseline accuracy
wandb.summary["baseline/accuracy"]      = round(baseline_acc_stats["p_hat"], 6)
wandb.summary["baseline/accuracy_ci"]   = [round(baseline_acc_stats["ci_lower"], 6),
                                            round(baseline_acc_stats["ci_upper"], 6)]
wandb.summary["baseline/accuracy_se"]   = round(baseline_acc_stats["se"], 6)
wandb.summary["baseline/accuracy_deff"] = round(baseline_acc_stats["deff"], 4)
wandb.summary["baseline/accuracy_n_eff"] = round(baseline_acc_stats["n_eff"], 2)
wandb.summary["baseline/accuracy_icc"]  = round(baseline_acc_stats["icc"], 4)

# Agent execution time
wandb.summary["agent/time_mean"]   = round(agent_time_stats["mean"], 4)
wandb.summary["agent/time_ci"]     = [round(agent_time_stats["ci_lower"], 4),
                                       round(agent_time_stats["ci_upper"], 4)]
wandb.summary["agent/time_se"]     = round(agent_time_stats["se"], 4)

# Baseline execution time
wandb.summary["baseline/time_mean"] = round(baseline_time_stats["mean"], 4)
wandb.summary["baseline/time_ci"]   = [round(baseline_time_stats["ci_lower"], 4),
                                        round(baseline_time_stats["ci_upper"], 4)]
wandb.summary["baseline/time_se"]   = round(baseline_time_stats["se"], 4)

# Agent time distribution
wandb.summary["agent/time_dist_preferred"]   = agent_dist["preferred"]
wandb.summary["agent/time_dist_gamma_aic"]   = round(agent_dist["gamma"]["aic"], 2)
wandb.summary["agent/time_dist_lognorm_aic"] = round(agent_dist["lognorm"]["aic"], 2)
wandb.summary["agent/time_dist_aic_diff"]    = round(agent_dist["aic_difference"], 2)

# Baseline time distribution
wandb.summary["baseline/time_dist_preferred"]   = baseline_dist["preferred"]
wandb.summary["baseline/time_dist_gamma_aic"]   = round(baseline_dist["gamma"]["aic"], 2)
wandb.summary["baseline/time_dist_lognorm_aic"] = round(baseline_dist["lognorm"]["aic"], 2)
wandb.summary["baseline/time_dist_aic_diff"]    = round(baseline_dist["aic_difference"], 2)

# ── Upload correctness matrices as W&B artifact ──────────────
artifact = wandb.Artifact(
    f"correctness-matrices-{wandb.run.id}",
    type="evaluation-data",
    description="Correctness matrices (N x m) for agent and baseline, plus time matrices. GSM8K evaluation.",
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(f.name,
        agent_correctness=agent_correctness_matrix,
        baseline_correctness=baseline_correctness_matrix,
        agent_times=agent_time_matrix,
        baseline_times=baseline_time_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print("✅ Correctness & time matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished. View your dashboard at the URL above.")


## 13. Performance Analysis & Next Steps

### What W&B gives you

Open the W&B run link to explore:

- **Latency panel** — per-agent and total pipeline latency across tasks, showing where time is spent.
- **Results table** — side-by-side text comparison of all four outputs (proposer, critic, refiner, baseline) for every task.
- **Overhead ratio** — how much more expensive the 3-agent pipeline is vs. the single-shot baseline.

### Architecture improvements enabled by LangChain

The LangChain refactor opens the door to several extensions without rewriting the core loop:

- **Multi-turn debate** — wrap the Proposer → Critic step in a loop with a LangChain `SequentialChain` or custom `RunnableSequence`.
- **Dynamic routing** — use a LangChain `RouterChain` to skip the Critic if the Proposer's confidence is high.
- **Tool-augmented agents** — give the Critic access to a calculator tool via `AgentExecutor` to verify arithmetic.
- **Model swaps** — replace the `HuggingFacePipeline` with `ChatOpenAI` or `ChatAnthropic` for API-based models without touching agent logic.

### Suggested next steps

1. **Automated benchmarking** — loop `run_multi_agent` over GSM8K / LogiQA splits and log accuracy to W&B.
2. **Hyperparameter sweeps** — use `wandb.sweep()` over temperature, top_p, and max_new_tokens.
3. **Model comparison** — run the same pipeline with Llama-3-8B or Gemma-7B and compare W&B dashboards.
4. **Human evaluation** — log outputs to a W&B `Table` with a "human_score" column and annotate post-hoc.
